# Libraries

In [1]:
import pandas as pd, matplotlib.pyplot as plt, numpy as np

# Dataset

In [2]:
df = pd.read_csv('../data/hotel_bookings.csv')
df

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,No Deposit,394.0,NaN,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,No Deposit,9.0,NaN,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,No Deposit,9.0,NaN,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,No Deposit,89.0,NaN,0,Transient,104.40,0,0,Check-Out,2017-09-07


# Dataset Overview

In this section, we examine the structure of the dataset to build a foundational understanding before deeper analysis. Specifically, we review: 

* the number of observations and features
* data types of each feature
* presence of missing values
* duplicate records

## Observations/features

In [3]:
df.shape

(119390, 32)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal                       

### Summary

* 32 features and 119,390 records

* 20 numerical features, and 12 categorical features

* Next step is to analyse missing values both in numerical and categorical 


## Missing Values

In [5]:
df.isnull().sum()

hotel                                  0
is_canceled                            0
lead_time                              0
arrival_date_year                      0
arrival_date_month                     0
arrival_date_week_number               0
arrival_date_day_of_month              0
stays_in_weekend_nights                0
stays_in_week_nights                   0
adults                                 0
children                               4
babies                                 0
meal                                   0
country                              488
market_segment                         0
distribution_channel                   0
is_repeated_guest                      0
previous_cancellations                 0
previous_bookings_not_canceled         0
reserved_room_type                     0
assigned_room_type                     0
booking_changes                        0
deposit_type                           0
agent                              16340
company         

`isnull` - illustrates four features has empty values.
Children, Country, Agent, and Company.

In [7]:
df.isnull().sum()[empty_features]/df.shape[0] * 100

children     0.003350
country      0.408744
agent       13.686238
company     94.306893
dtype: float64

In [6]:
empty_features = ['children', 'country', 'agent', 'company']
df.isnull().sum()[empty_features]/df.shape[0]

children    0.000034
country     0.004087
agent       0.136862
company     0.943069
dtype: float64

Above illustrates the % of the missing values in their respective feature.

Insights:
* Children: Minute amount, thus delete 
* Country: Minute amount, thus delete
* Company, and Agent: An overwhelming amount of missing values, therefore, requires investigation into it

In [60]:
# Build a function that will identify objects (categorical data types), and iterate over them with the unique function to see if there are hidden missing values as text

def display_categorical_data_type_as_unqiue(df):
    # retrieve the names under object data types
    columns = df.columns
    for x in columns:
        if df[x].dtypes == "str":
            print(x)
            print(df[x].unique())

display_categorical_data_type_as_unqiue(df)

hotel
<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str
arrival_date_month
<StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str
meal
<StringArray>
['BB', 'FB', 'HB', 'SC', 'Undefined']
Length: 5, dtype: str
country
<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA',   nan, 'ROU', 'NOR', 'OMN',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 178, dtype: str
market_segment
<StringArray>
[       'Direct',     'Corporate',     'Online TA', 'Offline TA/TO',
 'Complementary',        'Groups',     'Undefined',      'Aviation']
Length: 8, dtype: str
distribution_channel
<StringArray>
['Direct', 'Corporate', 'TA/TO', 'Undefined', 'GDS']
Length: 5, dtype: str
reserved_room_type
<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'P', 'B']
Length: 10, dtype: str
assigned_room_type
<StringArray>
['

The following categorical features have missing values:
* meal has undefined 
* market_segment has undefined
* distribution_channel has undefined

The next step here would be to calculate the total amount of missing values in the above listed features


In [61]:
def custom_sum_missing_values(df,feature):
    undefined = 0

    for x in df[feature]:
        if x == 'Undefined':
            undefined += 1
    percentage = undefined/len(df) * 100        
    print(f"{feature} undefined values: {undefined} : {percentage} %")
            

In [62]:
custom_sum_missing_values(df,'meal')
custom_sum_missing_values(df,'market_segment')
custom_sum_missing_values(df, 'distribution_channel')

meal undefined values: 1169 : 0.9791439819080325 %
market_segment undefined values: 2 : 0.0016751821760616465 %
distribution_channel undefined values: 5 : 0.004187955440154116 %


In [63]:
df.distribution_channel.value_counts()

distribution_channel
TA/TO        97870
Direct       14645
Corporate     6677
GDS            193
Undefined        5
Name: count, dtype: int64

In [64]:
df.market_segment.value_counts()

market_segment
Online TA        56477
Offline TA/TO    24219
Groups           19811
Direct           12606
Corporate         5295
Complementary      743
Aviation           237
Undefined            2
Name: count, dtype: int64

In [65]:
df.meal.value_counts()

meal
BB           92310
HB           14463
SC           10650
Undefined     1169
FB             798
Name: count, dtype: int64

* `meal` seems to have a reasonable amount of missing values, therefore investigation is needed there
* `market_segement` and `distribution_channel` have minute missing values, thus erase

## Investigations into missing values: Meal (Categorical Feature)

Code:
* BB - Bed & Breakfast
* HB - Half Board
* FB - Full Board
* SC - Self Catering

In [66]:
pd.crosstab(df['meal'],df['is_canceled'])

is_canceled,0,1
meal,,
BB,57800,34510
FB,320,478
HB,9479,4984
SC,6684,3966
Undefined,883,286


Hypothesis: Undefined may represent customers who did not select a meal plan and therefore could be behaviourally similar to Self-Catering (SC).

### $P(is\_cancelled | meal\ type)$

In [67]:
pd.crosstab(df['meal'],df['is_canceled'],normalize='index')

is_canceled,0,1
meal,,
BB,0.626151,0.373849
FB,0.401003,0.598997
HB,0.655397,0.344603
SC,0.627606,0.372394
Undefined,0.755346,0.244654


### $P( meal\ type | is\_cancelled )$

In [68]:
pd.crosstab(df['meal'],df['is_canceled'],normalize='columns')

is_canceled,0,1
meal,,
BB,0.768965,0.780346
FB,0.004257,0.010809
HB,0.126108,0.112699
SC,0.088923,0.089680
Undefined,0.011747,0.006467


### Meal Type resolution

Hypothesis: Undefined may represent customers who did not select a meal plan and therefore could be behaviourally similar to Self-Catering (SC).

To test this hypothesis, a conditional probability analysis was conducted using pd.crosstab, examining the cancellation rate across meal categories. The results showed that SC has a cancellation rate of approximately 37%, whereas Undefined has a significantly lower cancellation rate of approximately 24%.

This substantial difference suggests that Undefined does not follow the same behavioural pattern as SC. Therefore, merging these categories would result in the loss of a distinct predictive signal.

Although Undefined represents approximately 1% of the dataset (n = 1169), this sample size is sufficiently large to produce a stable probability estimate. Additionally, another category (FB) has an even smaller sample size, indicating that Undefined is not uniquely small within this feature

Based on this analysis, there is insufficient statistical justification to remove or merge the Undefined category. Instead, it will be retained and renamed to No_Meal to improve interpretability while preserving its predictive value.


## Investigations into missing values: Company, and Agent (Numerical Feature)

### Company
Company: ID of the company or cooperate that made the booking for this individual

In [69]:
df.company.isnull().sum()

np.int64(112593)

After doing research regarding company it turns out, company is actually the id of the company or cooperate that made the booking for this individual. Now, since most of the booking that's made are actually invdividuals, so regular customers. Which explains why there's so many missing values. Now, I could potentially recode the null values as non-cooperate/non-company, but this would later lead to sparseness, making it harder when it comes to the model for the models. Therefore, I have decided to drop the company feature itself. 

### Agent


Agent: List of ID's that corresponds to the travel agency

Hypothesis: Missing values could be directing booking (Customer booked directly)

In [70]:
df.agent.isnull().sum()

np.int64(16340)

In [71]:
from collections import Counter
Counter(df['distribution_channel'] == 'Direct')

Counter({False: 104745, True: 14645})

Hypothesis: Missing values could be directing booking (Customer booked directly)

The above hypothesis is clearly false. As Direct booking =/= missing values in the agent. Thus:
* Some non-direct bookings also have missing agent values
* OR direct bookings are not the only scenario where no agent exists

Missing agent values may indicate bookings where no identifiable travel agent was recorded, though this is not limited exclusively to direct booking channels. Therefore, replacing the null values with 0 allows these cases to be represented as bookings with no associated agent ID.

## Duplication

In [20]:
df.duplicated().sum()

np.int64(31994)

In [21]:
df[df.duplicated()]

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
5,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.00,0,1,Check-Out,2015-07-03
22,Resort Hotel,0,72,2015,July,27,1,2,4,2,...,No Deposit,250.0,NaN,0,Transient,84.67,0,1,Check-Out,2015-07-07
43,Resort Hotel,0,70,2015,July,27,2,2,3,2,...,No Deposit,250.0,NaN,0,Transient,137.00,0,1,Check-Out,2015-07-07
138,Resort Hotel,1,5,2015,July,28,5,1,0,2,...,No Deposit,240.0,NaN,0,Transient,97.00,0,0,Canceled,2015-07-01
200,Resort Hotel,0,0,2015,July,28,7,0,1,1,...,No Deposit,240.0,NaN,0,Transient,109.80,0,3,Check-Out,2015-07-08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119349,City Hotel,0,186,2017,August,35,31,0,3,2,...,No Deposit,9.0,NaN,0,Transient,126.00,0,2,Check-Out,2017-09-03
119352,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03
119353,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03
119354,City Hotel,0,63,2017,August,35,31,0,3,3,...,No Deposit,9.0,NaN,0,Transient-Party,195.33,0,2,Check-Out,2017-09-03


### Duplication Resolution 
It seems like there 31,994 duplicate records. Upon further examination there's no unique identifier that would uniquely identify that each record is made by the same person/individual. This means that, many individuals could've booked the same book details but two different individuals. Therefore, no actions will take place here.

# Handling missing data 

* Company: Erase the entire feature
* Agent: Replace empty values with a more useful insight value 0
* Country/Children: Minute amount, therefore erase the records
* Distribution Channel/ Market Segement: Erase the records that are missing
* meal replace undefined with no meal

In [22]:
len(df.columns)

32

In [23]:
# dropping the feature company
df_reduced = df.drop(columns=['company'])
len(df_reduced.columns)

31

In [24]:
len(df_reduced)

119390

In [25]:
# dropping the country, and column rows that are empty
df_reduced = df_reduced.dropna(subset=['country','children'])
len(df_reduced)

118898

In [26]:
# drop the rows from the feature: distribution channel and market segement, where there are undefined 

COLUMNS_TO_CLEAN = ["market_segment", "distribution_channel"]
VALUES_TO_REMOVE = ["Undefined"]

mask = df_reduced[COLUMNS_TO_CLEAN].isin(VALUES_TO_REMOVE).any(axis=1)
df_reduced = df_reduced.loc[~mask].copy()

In [27]:
len(df_reduced), len(df_reduced.columns)

(118897, 31)

In [28]:
df_reduced.agent.isnull().sum()

np.int64(16003)

In [29]:
# Agent we want to replace NaN with 0
df_reduced['agent'] = df_reduced['agent'].fillna(0)
df_reduced.agent.isnull().sum()

np.int64(0)

In [30]:
(df_reduced['meal'] == 'Undefined').sum()

np.int64(1165)

In [31]:
# meals that are undefined, replace with no meal

df_reduced['meal'] = df_reduced['meal'].replace("Undefined","No meal")
(df_reduced['meal'] == 'Undefined').sum()
(df_reduced['meal'] == 'No meal').sum()

np.int64(1165)

In [32]:
df_reduced

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,booking_changes,deposit_type,agent,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,3,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,4,No Deposit,0.0,0,Transient,0.00,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,No Deposit,0.0,0,Transient,75.00,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,No Deposit,304.0,0,Transient,75.00,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,No Deposit,240.0,0,Transient,98.00,0,1,Check-Out,2015-07-03
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
119385,City Hotel,0,23,2017,August,35,30,2,5,2,...,0,No Deposit,394.0,0,Transient,96.14,0,0,Check-Out,2017-09-06
119386,City Hotel,0,102,2017,August,35,31,2,5,3,...,0,No Deposit,9.0,0,Transient,225.43,0,2,Check-Out,2017-09-07
119387,City Hotel,0,34,2017,August,35,31,2,5,2,...,0,No Deposit,9.0,0,Transient,157.71,0,4,Check-Out,2017-09-07
119388,City Hotel,0,109,2017,August,35,31,2,5,2,...,0,No Deposit,89.0,0,Transient,104.40,0,0,Check-Out,2017-09-07


In [33]:
df_reduced.to_csv("../data/hotel_booking_clean.csv", index=False)

In [34]:
df_reduced.isnull().sum()

hotel                             0
is_canceled                       0
lead_time                         0
arrival_date_year                 0
arrival_date_month                0
arrival_date_week_number          0
arrival_date_day_of_month         0
stays_in_weekend_nights           0
stays_in_week_nights              0
adults                            0
children                          0
babies                            0
meal                              0
country                           0
market_segment                    0
distribution_channel              0
is_repeated_guest                 0
previous_cancellations            0
previous_bookings_not_canceled    0
reserved_room_type                0
assigned_room_type                0
booking_changes                   0
deposit_type                      0
agent                             0
days_in_waiting_list              0
customer_type                     0
adr                               0
required_car_parking_spaces 

In [35]:
display_categorical_data_type_as_unqiue(df_reduced)

hotel
<StringArray>
['Resort Hotel', 'City Hotel']
Length: 2, dtype: str
arrival_date_month
<StringArray>
[     'July',    'August', 'September',   'October',  'November',  'December',
   'January',  'February',     'March',     'April',       'May',      'June']
Length: 12, dtype: str
meal
<StringArray>
['BB', 'FB', 'HB', 'SC', 'No meal']
Length: 5, dtype: str
country
<StringArray>
['PRT', 'GBR', 'USA', 'ESP', 'IRL', 'FRA', 'ROU', 'NOR', 'OMN', 'ARG',
 ...
 'ATA', 'GTM', 'ASM', 'MRT', 'NCL', 'KIR', 'SDN', 'ATF', 'SLE', 'LAO']
Length: 177, dtype: str
market_segment
<StringArray>
[       'Direct',     'Corporate',     'Online TA', 'Offline TA/TO',
 'Complementary',        'Groups',      'Aviation']
Length: 7, dtype: str
distribution_channel
<StringArray>
['Direct', 'Corporate', 'TA/TO', 'GDS']
Length: 4, dtype: str
reserved_room_type
<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'H', 'L', 'B', 'P']
Length: 10, dtype: str
assigned_room_type
<StringArray>
['C', 'A', 'D', 'E', 'G', 'F', 'I'